# Phase 3 — Interpretability Analysis on the Two Production Checkpoints

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cerberus-neuro/blob/main/notebooks/06_phase_3_analysis.ipynb)

Runs the full argus-cells interpretability harness against the **two trained production checkpoints** from Phase 2 and produces the Phase 3 deliverables.

- **Argus-RN34** (`BaselineDiseaseClassifier`, 6-channel ResNet34) — Phase 2 best-epoch val_acc **0.82** (`patrickjreed/argus-rn34-v0`).
- **Argus-CCT** (`ArgusCCT`, from-scratch Compact Convolutional Transformer) — Phase 2 best-epoch val_acc **0.78** (`patrickjreed/argus-cct-v0`, `epoch_022.pt`).

**RN34 disambiguation:** Phase 2's RN34 training cell ran twice and each run's `epoch_NNN.pt` overwrote the previous on HF, so the on-Hub best epoch is unknown. This notebook downloads every available `epoch_NNN.pt`, scores each on the balanced val batch, and keeps the best-scoring checkpoint (expect ~0.82). Argus-CCT is a single clean run, so `epoch_022.pt` is loaded directly.

**Four deliverables (printed in the final summary cell):**
1. Per-(cell_type, channel) top channels per model (cell-type-stratified channel ablation).
2. Cross-architecture agreement: IG-vs-IG per-sample saliency agreement + Spearman of the two models' global channel-ablation rankings.
3. Donor / disease probe ratio per model with the OK / RED-FLAG interpretation.
4. A clearly-labeled TODO line for the human-written biology claim.

**Spec:** `docs/superpowers/specs/2026-05-12-argus-cells-design.md` §4-§6. **Predecessor:** Phase 2 paired training (RN34 0.82, CCT 0.78).

In [ ]:
# Install the argus_cells package code only (--no-deps) so pip never touches
# Colab's pre-built numpy / scipy / scikit-learn. Upgrading those mid-session
# breaks the numpy<->scipy ABI (the "_blas_supports_fpe" / "_center" ImportErrors).
!pip install -q --upgrade --no-deps git+https://github.com/PatrickJReed/argus-cells.git@main
# Only the two deps Colab may lack. Both are pure-Python and pull NO numpy-linked
# packages, so the scientific stack stays exactly as Colab shipped it. This
# notebook deliberately does NOT import argus_cells.training (avoids the
# pytorch_msssim dependency); accuracy is computed inline from argmax vs labels.
!pip install -q boto3 huggingface_hub

import sys
for _m in list(sys.modules):
    if _m == 'argus_cells' or _m.startswith('argus_cells.'):
        del sys.modules[_m]
print('install + cache purge done')

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# HF login + Drive mount (mirror nb04/nb05). On Colab the TIFF cache lives on the
# fast local SSD; off Colab everything falls back to a local cache dir.
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('HF login: OK (Colab secret)')
except Exception as e:
    print(f'HF login skipped ({type(e).__name__}); set HF_TOKEN if downloads 401')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path('/content/cerberus-cache')  # fast local SSD; reuses the Phase 2 cache
except Exception:
    CACHE_DIR = Path.home() / '.cache' / 'cerberus-neuro' / 'v0'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'CACHE_DIR: {CACHE_DIR}')
print(f'device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"})')

from argus_cells.data import (
    build_manifest, subset_manifest, well_level_split, NeuroPaintingDataset,
    CELL_TYPES, LINE_CONDITIONS,
)
from argus_cells.model import BaselineDiseaseClassifier
from argus_cells.models.cct import ArgusCCT
from argus_cells.attribution import (
    compute_channel_ablation,
    compute_channel_ablation_per_sample,
    compute_gradcam,
    compute_integrated_gradients,
    compute_attention_rollout,
)
from argus_cells.probes import parallel_probe_report
from argus_cells.analysis import (
    plot_channel_ablation_heatmap,
    plot_probe_comparison,
    stratify_channel_scores_by_cell_type,
    saliency_agreement,
)

# Channel order is fixed by the data pipeline: brightfield then 5 fluorescence.
CHANNEL_NAMES = ['BF', 'DNA', 'Mito', 'AGP', 'ER', 'RNA']
CELL_TYPE_NAMES = ['stem', 'progen', 'neuron', 'astro']

In [ ]:
# Same held-out val split as Phase 2 training: build_manifest(CACHE_DIR) on the
# default batches, subset_manifest(48 wells/cell type, 9 sites/well, seed=0),
# well_level_split(val_frac=0.2, seed=0). We interpret only the val split.
WELLS_PER_CELL_TYPE = 48
SITES_PER_WELL = 9

manifest = build_manifest(CACHE_DIR)
subset = subset_manifest(manifest, wells_per_cell_type=WELLS_PER_CELL_TYPE,
                         sites_per_well=SITES_PER_WELL, seed=0)
train_manifest, val_manifest = well_level_split(subset, val_frac=0.2, seed=0)
print(f'train sites: {len(train_manifest):,}')
print(f'val sites:   {len(val_manifest):,}')

In [ ]:
# CELL-TYPE-BALANCED val batch for the attribution methods (mirrors nb04 cell 7).
# val_manifest is grouped by cell type, so a first-512 pass over a shuffle=False
# loader returns essentially one cell type (others show n_samples=0 in the
# stratification table). Instead bucket by (cell_type, line_condition) from a
# SHUFFLED pass and stop once every one of the 8 groups has TARGET_PER_GROUP
# crops, so the global ablation and the per-cell-type stratification are
# representative across all 4 cell types and both conditions.
TARGET_PER_GROUP = 32  # 256 crops total; keeps the CCT tokenizer (128ch @ 256px) within an L4

collect_dataset = NeuroPaintingDataset(
    manifest=val_manifest,
    cache_dir=CACHE_DIR,
    crop_size=256,
    crops_per_site=10,
    min_cells_per_crop=3,
    shuffle=True,  # mix cell types so all 8 (cell_type, condition) groups fill together
    augment=False,
)
collect_loader = torch.utils.data.DataLoader(
    collect_dataset, batch_size=32, num_workers=4, persistent_workers=False
)

buckets = {
    (c, d): {'bf': [], 'fluo': [], 'ct': [], 'cond': []}
    for c in range(len(CELL_TYPES))
    for d in range(len(LINE_CONDITIONS))
}


def groups_full():
    return all(len(b['bf']) >= TARGET_PER_GROUP for b in buckets.values())


# Loop ends either when all 8 groups hit TARGET_PER_GROUP or the loader is
# exhausted (a sparse group simply contributes however many crops exist).
for bf, fluo, ct, cond in collect_loader:
    for j in range(bf.shape[0]):
        b = buckets[(int(ct[j]), int(cond[j]))]
        if len(b['bf']) < TARGET_PER_GROUP:
            b['bf'].append(bf[j])
            b['fluo'].append(fluo[j])
            b['ct'].append(ct[j])
            b['cond'].append(cond[j])
    if groups_full():
        break

val_bf = torch.stack([t for b in buckets.values() for t in b['bf']])
val_fluo = torch.stack([t for b in buckets.values() for t in b['fluo']])
val_ct = torch.stack([t for b in buckets.values() for t in b['ct']])
val_cond = torch.stack([t for b in buckets.values() for t in b['cond']])

# Deterministic shuffle so the GradCAM / rollout / IG subset (val_images[:32]) is
# also mixed, not all one (cell_type, condition) group.
perm = torch.randperm(val_ct.shape[0], generator=torch.Generator().manual_seed(0))
val_bf, val_fluo, val_ct, val_cond = val_bf[perm], val_fluo[perm], val_ct[perm], val_cond[perm]

val_images = torch.cat([val_bf, val_fluo], dim=1).to(DEVICE)  # [<=512, 6, 256, 256]
val_labels = val_cond.to(DEVICE)

print(f'collected val batch: {val_images.shape}')
for c in range(len(CELL_TYPES)):
    counts = {LINE_CONDITIONS[d]: len(buckets[(c, d)]['bf']) for d in range(len(LINE_CONDITIONS))}
    print(f'  {CELL_TYPES[c]:>6}: {counts}')
print(f'cell_type distribution: {torch.bincount(val_ct, minlength=len(CELL_TYPES)).tolist()}')
print(f'condition distribution: {torch.bincount(val_cond, minlength=len(LINE_CONDITIONS)).tolist()}')

In [ ]:
# Load + select the two production checkpoints. Accuracy is computed INLINE
# (argmax vs labels) so we never import argus_cells.training.
from huggingface_hub import hf_hub_download, list_repo_files

RN34_REPO = 'patrickjreed/argus-rn34-v0'
CCT_REPO = 'patrickjreed/argus-cct-v0'


@torch.no_grad()
def val_batch_accuracy(model, chunk=64):
    """Inline disease accuracy on the balanced val batch (argmax vs labels),
    forwarded in chunks so the CCT tokenizer (128ch @ 256px) never OOMs."""
    correct = 0
    for i in range(0, val_images.shape[0], chunk):
        logits = model(val_images[i:i + chunk])
        correct += (logits.argmax(-1) == val_labels[i:i + chunk]).sum().item()
    return correct / val_images.shape[0]


# --- Argus-RN34 disambiguation ---------------------------------------------
# Phase 2's RN34 cell ran twice; each run's epoch_NNN.pt overwrote the previous
# on HF, so the on-Hub best epoch is unknown. Download every available
# epoch_NNN.pt, load each into a fresh BaselineDiseaseClassifier, score it on the
# balanced val batch, and KEEP THE BEST-SCORING checkpoint (expect best ~0.82).
rn34_epoch_files = sorted(
    f for f in list_repo_files(RN34_REPO)
    if f.startswith('epoch_') and f.endswith('.pt')
)
print(f'{RN34_REPO} epoch checkpoints found: {rn34_epoch_files}')

rn34_scores = {}
rn34 = None
rn34_best_file = None
rn34_best_acc = -1.0
for fname in rn34_epoch_files:
    ckpt_path = hf_hub_download(RN34_REPO, fname)
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    candidate = BaselineDiseaseClassifier(in_channels=6, n_classes=2, pretrained_encoder=False)
    candidate.load_state_dict(ckpt['model'])
    candidate = candidate.to(DEVICE).eval()
    acc = val_batch_accuracy(candidate)
    rn34_scores[fname] = acc
    print(f'  {fname}: val-batch acc = {acc:.4f}')
    if acc > rn34_best_acc:
        rn34_best_acc, rn34_best_file, rn34 = acc, fname, candidate

print()
print(f'>>> selected Argus-RN34 checkpoint: {rn34_best_file}  (val-batch acc {rn34_best_acc:.4f})')

# Free the GPU memory the 15-checkpoint sweep reserved before the CCT forward
# (the CCT tokenizer is memory-heavy at full 256px resolution).
del candidate, ckpt
import gc
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

# --- Argus-CCT (single clean run; epoch_022.pt is its best) -----------------
cct_path = hf_hub_download(CCT_REPO, 'epoch_022.pt')
cct_ckpt = torch.load(cct_path, map_location='cpu', weights_only=False)
cct = ArgusCCT(in_channels=6, n_classes=2, img_size=256)
cct.load_state_dict(cct_ckpt['model'])
cct = cct.to(DEVICE).eval()
cct_acc = val_batch_accuracy(cct)
print(f'>>> Argus-CCT epoch_022.pt: val-batch acc {cct_acc:.4f}  (expect ~0.78)')

## § A — Argus-RN34 harness

Channel ablation (global + per-sample), cell-type-stratified channel ablation, **GradCAM** (CNN-specific, on `encoder.layer4`), and Integrated Gradients. Results stored under the `rn34_*` names for the cross-architecture comparison and the summary block.

In [ ]:
# Channel ablation: per-channel accuracy drop over the whole balanced val batch,
# plus per-sample per-channel confidence drop for the cell-type stratification.
rn34_ablation = compute_channel_ablation(model=rn34, images=val_images, labels=val_labels)
print('Argus-RN34 per-channel accuracy drop (whole val batch):')
for i, ch in enumerate(CHANNEL_NAMES):
    print(f'  {ch:>4}: {rn34_ablation.channel_scores[i].item():+.4f}')
print(f'baseline accuracy: {rn34_ablation.metadata["baseline_accuracy"]:.4f}')
print()

rn34_ablation_ps = compute_channel_ablation_per_sample(
    model=rn34, images=val_images, labels=val_labels,
)
print(f'per-sample channel_scores shape: {rn34_ablation_ps.channel_scores.shape}')

rn34_strat = stratify_channel_scores_by_cell_type(
    result=rn34_ablation_ps,
    cell_types=val_ct,
    cell_type_names=CELL_TYPE_NAMES,
    channel_names=CHANNEL_NAMES,
)
print(rn34_strat.pivot(index='cell_type', columns='channel', values='mean_score').to_string())

In [ ]:
# GradCAM + IG on the first 32 val crops (full saliency maps are heavy; a small
# representative subset is enough for the figures and the agreement comparison).
subset_n = 32
rn34_gradcam = compute_gradcam(
    model=rn34,
    target_layer=rn34.encoder.layer4,
    images=val_images[:subset_n],
    target_class=1,  # disease (deletion) class
)
print(f'RN34 GradCAM saliency shape: {rn34_gradcam.saliency.shape}')
print(f'RN34 GradCAM channel_scores shape: {rn34_gradcam.channel_scores.shape}')

rn34_ig = compute_integrated_gradients(
    model=rn34,
    images=val_images[:subset_n],
    target_class=1,
    n_steps=32,
)
print(f'RN34 IG saliency shape: {rn34_ig.saliency.shape}')
print('RN34 IG per-channel |sum| (mean over 32 samples):')
for i, ch in enumerate(CHANNEL_NAMES):
    print(f'  {ch:>4}: {rn34_ig.channel_scores[:, i].mean().item():+.3f}')

## § B — Argus-CCT harness

Same channel ablation + cell-type stratification. The CNN-specific GradCAM is replaced by **attention rollout** (transformer-specific, consumes `cct.attention_maps`); Integrated Gradients runs identically (architecture-agnostic). Results stored under the `cct_*` names.

In [ ]:
cct_ablation = compute_channel_ablation(model=cct, images=val_images, labels=val_labels)
print('Argus-CCT per-channel accuracy drop (whole val batch):')
for i, ch in enumerate(CHANNEL_NAMES):
    print(f'  {ch:>4}: {cct_ablation.channel_scores[i].item():+.4f}')
print(f'baseline accuracy: {cct_ablation.metadata["baseline_accuracy"]:.4f}')
print()

cct_ablation_ps = compute_channel_ablation_per_sample(
    model=cct, images=val_images, labels=val_labels,
)
print(f'per-sample channel_scores shape: {cct_ablation_ps.channel_scores.shape}')

cct_strat = stratify_channel_scores_by_cell_type(
    result=cct_ablation_ps,
    cell_types=val_ct,
    cell_type_names=CELL_TYPE_NAMES,
    channel_names=CHANNEL_NAMES,
)
print(cct_strat.pivot(index='cell_type', columns='channel', values='mean_score').to_string())

In [ ]:
# Attention rollout (transformer-specific) instead of GradCAM, plus IG on the
# same 32-crop subset. Rollout is class-agnostic; target_class=1 is recorded in
# metadata only (kept for interface symmetry with the gradient methods).
cct_rollout = compute_attention_rollout(cct, val_images[:subset_n], target_class=1)
print(f'CCT attention-rollout saliency shape: {cct_rollout.saliency.shape}')
print(f'CCT attention-rollout channel_scores shape: {cct_rollout.channel_scores.shape}')

cct_ig = compute_integrated_gradients(
    model=cct,
    images=val_images[:subset_n],
    target_class=1,
    n_steps=32,
)
print(f'CCT IG saliency shape: {cct_ig.saliency.shape}')
print('CCT IG per-channel |sum| (mean over 32 samples):')
for i, ch in enumerate(CHANNEL_NAMES):
    print(f'  {ch:>4}: {cct_ig.channel_scores[:, i].mean().item():+.3f}')

## Donor confound probe — both models

Reuses the nb04 `yield_donor` pattern: build train + val `yield_donor=True` loaders once, define `collect_embeddings`, run it for each model's frozen encoder, then fit two parallel probes (donor identity + disease) per model. A donor/disease ratio ≪ 1 is good (the encoder retained little donor info); ≥ 1 is a red flag that the disease number may ride donor identity rather than disease biology.

In [ ]:
# yield_donor=True makes the loader emit (bf, fluo, cell_type, line_condition,
# donor) so embedding i keeps its TRUE donor i (no positional approximation).
train_donor_dataset = NeuroPaintingDataset(
    manifest=train_manifest,
    cache_dir=CACHE_DIR,
    crop_size=256,
    crops_per_site=10,
    min_cells_per_crop=3,
    shuffle=True,
    augment=False,
    yield_donor=True,
)
val_donor_dataset = NeuroPaintingDataset(
    manifest=val_manifest,
    cache_dir=CACHE_DIR,
    crop_size=256,
    crops_per_site=10,
    min_cells_per_crop=3,
    shuffle=True,
    augment=False,
    yield_donor=True,
)
train_donor_loader = torch.utils.data.DataLoader(
    train_donor_dataset, batch_size=32, num_workers=4, persistent_workers=False
)
val_donor_loader = torch.utils.data.DataLoader(
    val_donor_dataset, batch_size=32, num_workers=4, persistent_workers=False
)


@torch.no_grad()
def collect_embeddings(loader, model, n_samples):
    """Iterate a yield_donor=True loader, returning embeddings paired with their
    true disease and donor labels. embedding i, disease i, and donor i all
    correspond to the same crop (no positional approximation). Both models expose
    extract_embedding(x) -> [B, D] (RN34: 512-dim pooled conv; CCT: pooled token)."""
    embs, diseases, donors = [], [], []
    collected = 0
    for bf, fluo, ct, cond, donor in loader:
        x = torch.cat([bf, fluo], dim=1).to(DEVICE)
        embs.append(model.extract_embedding(x).cpu().numpy())
        diseases.append(cond.cpu().numpy())
        donors.append(donor.cpu().numpy())
        collected += bf.shape[0]
        if collected >= n_samples:
            break
    return (
        np.concatenate(embs)[:n_samples],
        np.concatenate(diseases)[:n_samples],
        np.concatenate(donors)[:n_samples],
    )


def run_probe(model):
    """Collect 1024 train + 256 val embeddings for one model and fit the two
    parallel probes. Donor IDs are re-indexed to contiguous [0, n_donors)."""
    tr_emb, tr_disease, tr_donor_raw = collect_embeddings(train_donor_loader, model, n_samples=1024)
    va_emb, va_disease, va_donor_raw = collect_embeddings(val_donor_loader, model, n_samples=256)
    all_donors = sorted(set(tr_donor_raw.tolist()) | set(va_donor_raw.tolist()))
    donor_to_idx = {d: i for i, d in enumerate(all_donors)}
    tr_donor = np.array([donor_to_idx[d] for d in tr_donor_raw], dtype='int64')
    va_donor = np.array([donor_to_idx[d] for d in va_donor_raw], dtype='int64')
    n_donors = len(all_donors)
    report = parallel_probe_report(
        train_emb=tr_emb, train_donor=tr_donor, train_disease=tr_disease.astype('int64'),
        val_emb=va_emb, val_donor=va_donor, val_disease=va_disease.astype('int64'),
        n_donors=n_donors,
    )
    return report, n_donors


rn34_probe, rn34_n_donors = run_probe(rn34)
cct_probe, cct_n_donors = run_probe(cct)

for name, report, n_donors in [
    ('Argus-RN34', rn34_probe, rn34_n_donors),
    ('Argus-CCT', cct_probe, cct_n_donors),
]:
    print(f'{name} (N donors observed: {n_donors}):')
    print(f"  donor   val_acc: {report['donor']['val_accuracy']:.4f} "
          f"(random {report['donor']['random_baseline']:.4f})")
    print(f"  disease val_acc: {report['disease']['val_accuracy']:.4f} "
          f"(random {report['disease']['random_baseline']:.4f})")
    print(f"  ratio (donor/disease): {report['ratio']:.3f}")
    print()

## § C — Cross-architecture agreement

Two like-for-like comparisons between Argus-RN34 and Argus-CCT:

1. **IG-vs-IG saliency agreement.** Both IG saliency maps are `[B, 6, H, W]` (same shape), the valid pixel-level like-for-like comparison. (GradCAM `[B,1,H,W]` vs rollout `[B,1,H,W]` are *different methods*, so they are not compared; channel ablation has `saliency=None` and must not be passed to `saliency_agreement`.)
2. **Global channel-ablation ranking agreement.** Spearman correlation of the two models' 6 global channel-drop scores, printed side by side.

In [ ]:
# 1) IG-vs-IG per-sample saliency agreement (both [B, 6, H, W] — valid like-for-like).
ig_agreement = saliency_agreement(rn34_ig, cct_ig)
print(f'IG-vs-IG saliency agreement (per-sample Spearman):')
print(f'  mean over {ig_agreement["per_sample"].shape[0]} samples: {ig_agreement["mean"]:.4f}')
print()

# 2) Global channel-ablation ranking agreement. Do NOT pass the ablation results
# to saliency_agreement (they carry saliency=None); compare the 6 channel-drop
# scores directly with Spearman.
rn34_drops = rn34_ablation.channel_scores.cpu().numpy()
cct_drops = cct_ablation.channel_scores.cpu().numpy()
rn34_rank = [CHANNEL_NAMES[i] for i in rn34_drops.argsort()[::-1]]
cct_rank = [CHANNEL_NAMES[i] for i in cct_drops.argsort()[::-1]]

print('Global channel-ablation ranking (highest accuracy drop first):')
print(f'  Argus-RN34: {rn34_rank}')
print(f'  Argus-CCT:  {cct_rank}')
print()
print('Per-channel drop, side by side:')
print(f'  {"channel":>8} {"RN34":>9} {"CCT":>9}')
for i, ch in enumerate(CHANNEL_NAMES):
    print(f'  {ch:>8} {rn34_drops[i]:>+9.4f} {cct_drops[i]:>+9.4f}')

channel_rank_rho, channel_rank_p = spearmanr(rn34_drops, cct_drops)
print()
print(f'channel-ranking Spearman (RN34 vs CCT, 6 channels): rho={channel_rank_rho:.4f}, p={channel_rank_p:.4f}')

In [ ]:
# Production figures: channel-ablation heatmap + probe comparison, per model.
for name, strat, report, n_donors in [
    ('Argus-RN34', rn34_strat, rn34_probe, rn34_n_donors),
    ('Argus-CCT', cct_strat, cct_probe, cct_n_donors),
]:
    plot_channel_ablation_heatmap(
        df=strat,
        cell_type_order=CELL_TYPE_NAMES,
        channel_order=CHANNEL_NAMES,
        title=f'Channel-ablation confidence drop per (cell type, channel) — {name}',
    )
    plt.show()

    plot_probe_comparison(
        report=report,
        title=f'Donor probe (N={n_donors}) vs disease probe (N=2) — {name}',
    )
    plt.show()

In [ ]:
# Summary block for the Phase 3 results report — the four deliverables.
def top_channels_per_cell_type(strat_df, k=2):
    lines = []
    for ct in CELL_TYPE_NAMES:
        sub = strat_df[strat_df['cell_type'] == ct].sort_values('mean_score', ascending=False).head(k)
        pairs = ', '.join(f'{r.channel}={r.mean_score:+.3f}' for r in sub.itertuples())
        lines.append(f'    {ct:>6}: {pairs}')
    return '\n'.join(lines)


def ratio_verdict(ratio):
    return (
        'RED FLAG — donor info is at least as extractable as disease info'
        if ratio >= 1.0
        else 'OK — encoder retains less donor info than disease info'
    )


print('=' * 64)
print('PHASE 3 INTERPRETABILITY RESULT SUMMARY')
print('=' * 64)
print()
print(f'Selected checkpoints:')
print(f'  Argus-RN34: {rn34_best_file}  (val-batch acc {rn34_best_acc:.4f})')
print(f'  Argus-CCT:  epoch_022.pt  (val-batch acc {cct_acc:.4f})')
print()
print('-' * 64)
print('DELIVERABLE (a) — per-(cell_type, channel) top channels per model')
print('-' * 64)
print('  Argus-RN34 (channel-ablation confidence drop, top-2 per cell type):')
print(top_channels_per_cell_type(rn34_strat))
print('  Argus-CCT (channel-ablation confidence drop, top-2 per cell type):')
print(top_channels_per_cell_type(cct_strat))
print()
print('-' * 64)
print('DELIVERABLE (b) — cross-architecture agreement')
print('-' * 64)
print(f'  IG-vs-IG per-sample saliency agreement (mean Spearman): {ig_agreement["mean"]:.4f}')
print(f'  Global channel-ablation ranking:')
print(f'    Argus-RN34: {rn34_rank}')
print(f'    Argus-CCT:  {cct_rank}')
print(f'  Channel-ranking Spearman (6 channels): rho={channel_rank_rho:.4f} (p={channel_rank_p:.4f})')
print()
print('-' * 64)
print('DELIVERABLE (c) — donor / disease probe ratio per model')
print('-' * 64)
for name, report, n_donors in [
    ('Argus-RN34', rn34_probe, rn34_n_donors),
    ('Argus-CCT', cct_probe, cct_n_donors),
]:
    print(f'  {name} (N donors observed: {n_donors}):')
    print(f"    donor val_acc:   {report['donor']['val_accuracy']:.4f}  "
          f"(random {report['donor']['random_baseline']:.4f})")
    print(f"    disease val_acc: {report['disease']['val_accuracy']:.4f}  (random 0.5)")
    print(f"    ratio (donor/disease): {report['ratio']:.3f}")
    print(f'    interpretation: {ratio_verdict(report["ratio"])}')
print()
print('-' * 64)
print('DELIVERABLE (d) — biology claim')
print('-' * 64)
print('  TODO (human-written): interpret the per-cell-type channel rankings above')
print('  against the Tegtmeyer 2025 mitochondrial-structure finding in 22q11.2')
print('  deletion astrocytes. Do NOT auto-generate this claim; it requires a')
print('  human reading the astro-row Mito ranking + the donor-probe verdict and')
print('  deciding what is honestly supportable. Numbers above are the inputs.')